# Differential Expression (DESeq2) - headless

Reproduces a paper's DEG finding from a gene-count matrix.


In [ ]:
# Parameters (injected at launch). All are str/number so injection stays valid R.
counts_path <- "/data/counts.tsv"           # salmon.merged.gene_counts.tsv (gene_id + per-sample columns)
output_path <- "/outputs/de_results.csv"
id_column <- "gene_id"
test_samples <- ""                            # comma-separated sample column names (treatment)
reference_samples <- ""                       # comma-separated sample column names (control)
block_labels <- ""                            # optional matched-pairs: per-sample block/subject
                                              # labels, comma-separated, ALIGNED to c(test,reference)
lfc_threshold <- 1.0
padj_threshold <- 0.05


In [ ]:
suppressMessages(library(DESeq2))

test_s <- trimws(strsplit(test_samples, ",")[[1]]); test_s <- test_s[test_s != ""]
ref_s  <- trimws(strsplit(reference_samples, ",")[[1]]); ref_s <- ref_s[ref_s != ""]
stopifnot(length(test_s) > 0, length(ref_s) > 0)
samples <- c(test_s, ref_s)
condition <- factor(c(rep("test", length(test_s)), rep("reference", length(ref_s))), levels = c("reference", "test"))
coldata <- data.frame(condition = condition, row.names = samples)

# Matched-pairs / blocked design: when a block label is supplied for EVERY sample and there are
# >= 2 distinct labels, model `~ block + condition` so donor-to-donor baseline variance is removed
# (sharply raising power for a paired study). Otherwise fall back to the unpaired `~ condition`.
block_s <- trimws(strsplit(block_labels, ",")[[1]]); block_s <- block_s[block_s != ""]
use_block <- length(block_s) == length(samples) && length(unique(block_s)) >= 2
if (use_block) {
  coldata$block <- factor(block_s)
  design_formula <- ~ block + condition
} else {
  design_formula <- ~ condition
}

mat <- read.delim(counts_path, check.names = FALSE, stringsAsFactors = FALSE)
if (!(id_column %in% colnames(mat))) id_column <- colnames(mat)[1]
rownames(mat) <- make.unique(as.character(mat[[id_column]]))
missing <- setdiff(samples, colnames(mat))
if (length(missing) > 0) stop(paste("samples not in matrix:", paste(missing, collapse=", ")))
counts <- as.matrix(mat[, samples, drop = FALSE])
counts <- matrix(as.integer(round(as.numeric(counts))), nrow = nrow(counts), dimnames = dimnames(counts))

dds <- DESeqDataSetFromMatrix(countData = counts, colData = coldata, design = design_formula)
dds <- DESeq(dds)
res <- as.data.frame(results(dds, contrast = c("condition", "test", "reference")))
dir.create(dirname(output_path), showWarnings = FALSE, recursive = TRUE)
out <- data.frame(gene_id = rownames(res), log2FoldChange = res$log2FoldChange, padj = res$padj)
write.csv(out, output_path, row.names = FALSE)
cat("wrote", nrow(out), "genes to", output_path, "\n")
if (use_block) cat("design: ~ block + condition (paired,", length(unique(block_s)), "subjects)\n")
